In [1]:
# ============================================
# IPL Win Probability Predictor
# Day 5 — Model Training
# Phase 6 — Encoding, Splitting, Training, Evaluating
# ============================================
# GOAL: One-hot encode teams, split data, train 
# Random Forest and XGBoost, compare accuracy

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import xgboost as xgb

model_df = pd.read_csv("../data/model_ready_data.csv")
print("✅ Loaded model-ready data")
print("Shape:", model_df.shape)
model_df.head()

✅ Loaded model-ready data
Shape: (83480, 10)


,match_id,batting_team,bowling_team,current_score,wicket_fallen,balls_remaining,runs_remaining,current_run_rate,required_run_rate,batting_team_won
0,1,Royal Challengers Bangalore,Sunrisers Hyderabad,1,0,119,207,6.0,10.436975,0
1,1,Royal Challengers Bangalore,Sunrisers Hyderabad,1,0,118,207,3.0,10.525424,0
2,1,Royal Challengers Bangalore,Sunrisers Hyderabad,1,0,117,207,2.0,10.615385,0
3,1,Royal Challengers Bangalore,Sunrisers Hyderabad,3,0,116,205,4.5,10.603448,0
4,1,Royal Challengers Bangalore,Sunrisers Hyderabad,7,0,115,201,8.4,10.486957,0


In [2]:
# One-hot encode batting_team and bowling_team
model_df_encoded=pd.get_dummies(model_df,columns=['batting_team','bowling_team'])

print("Shape before encoding:", model_df.shape)
print("Shape after encoding:", model_df_encoded.shape)
model_df_encoded.head()

Shape before encoding: (83480, 10)
Shape after encoding: (83480, 34)


,match_id,current_score,wicket_fallen,balls_remaining,runs_remaining,current_run_rate,required_run_rate,batting_team_won,batting_team_Chennai Super Kings,batting_team_Deccan Chargers,...,bowling_team_Gujarat Lions,bowling_team_Kings XI Punjab,bowling_team_Kochi Tuskers Kerala,bowling_team_Kolkata Knight Riders,bowling_team_Mumbai Indians,bowling_team_Pune Warriors,bowling_team_Rajasthan Royals,bowling_team_Rising Pune Supergiants,bowling_team_Royal Challengers Bangalore,bowling_team_Sunrisers Hyderabad
0,1,1,0,119,207,6.0,10.436975,0,False,False,...,False,False,False,False,False,False,False,False,False,True
1,1,1,0,118,207,3.0,10.525424,0,False,False,...,False,False,False,False,False,False,False,False,False,True
2,1,1,0,117,207,2.0,10.615385,0,False,False,...,False,False,False,False,False,False,False,False,False,True
3,1,3,0,116,205,4.5,10.603448,0,False,False,...,False,False,False,False,False,False,False,False,False,True
4,1,7,0,115,201,8.4,10.486957,0,False,False,...,False,False,False,False,False,False,False,False,False,True


In [3]:
# X = everything the model uses to predict (features)
# y = what we want to predict (target)

X=model_df_encoded.drop(columns=['match_id','batting_team_won'])
y=model_df_encoded['batting_team_won']

print("Features shape (X):", X.shape)
print("Target shape (y):", y.shape)
print("\nFeature columns:", X.columns.tolist()[:10], "...")

Features shape (X): (83480, 32)
Target shape (y): (83480,)

Feature columns: ['current_score', 'wicket_fallen', 'balls_remaining', 'runs_remaining', 'current_run_rate', 'required_run_rate', 'batting_team_Chennai Super Kings', 'batting_team_Deccan Chargers', 'batting_team_Delhi Daredevils', 'batting_team_Gujarat Lions'] ...


In [4]:
# Get unique match IDs
unique_matches = model_df_encoded['match_id'].unique()
print("Total unique matches:", len(unique_matches))

# Split MATCH IDs (not rows) into train and test
train_match_ids, test_match_ids = train_test_split(
    unique_matches, 
    test_size=0.2, 
    random_state=42
)

print("Matches in train:", len(train_match_ids))
print("Matches in test:", len(test_match_ids))

# Now select ALL rows belonging to train matches, and ALL rows belonging to test matches
train_data = model_df_encoded[model_df_encoded['match_id'].isin(train_match_ids)]
test_data = model_df_encoded[model_df_encoded['match_id'].isin(test_match_ids)]

# Separate features and target for each
X_train = train_data.drop(columns=['match_id', 'batting_team_won'])
y_train = train_data['batting_team_won']

X_test = test_data.drop(columns=['match_id', 'batting_team_won'])
y_test = test_data['batting_team_won']

print("\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

# Critical verification - this should now be ZERO
overlap_check = set(train_data['match_id']) & set(test_data['match_id'])
print("\nMatch overlap between train/test:", len(overlap_check))

Total unique matches: 724
Matches in train: 579
Matches in test: 145

Training rows: 66968
Testing rows: 16512

Match overlap between train/test: 0


In [5]:
# Train Random Forest
rf_model=RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

rf_predictions=rf_model.predict(X_test)
rf_accuracy=accuracy_score(y_test,rf_predictions)

print(f"✅ Random Forest trained")
print(f"Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")

✅ Random Forest trained
Accuracy: 0.7604 (76.04%)


In [6]:
# Train XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_predictions)

print(f"✅ XGBoost trained")
print(f"Accuracy: {xgb_accuracy:.4f} ({xgb_accuracy*100:.2f}%)")

✅ XGBoost trained
Accuracy: 0.7713 (77.13%)


In [7]:
# Compare Both Models
print("MODEL COMPARISON")
print("-" * 30)
print(f"Random Forest Accuracy: {rf_accuracy*100:.2f}%")
print(f"XGBoost Accuracy:       {xgb_accuracy*100:.2f}%")

if xgb_accuracy > rf_accuracy:
    print("\n🏆 XGBoost performs better")
    best_model = xgb_model
    best_model_name = "XGBoost"
else:
    print("\n🏆 Random Forest performs better")
    best_model = rf_model
    best_model_name = "Random Forest"

MODEL COMPARISON
------------------------------
Random Forest Accuracy: 76.04%
XGBoost Accuracy:       77.13%

🏆 XGBoost performs better


In [8]:
# Detailed Evaluation of Best Model
best_predictions=best_model.predict(X_test)

print(f"Detailed Evaluation — {best_model_name}")
print("-" * 40)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test,best_predictions))

print("\nClassification Report:")
print(classification_report(y_test, best_predictions))

Detailed Evaluation — XGBoost
----------------------------------------

Confusion Matrix:
[[5512 1685]
 [2092 7223]]

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.77      0.74      7197
           1       0.81      0.78      0.79      9315

    accuracy                           0.77     16512
   macro avg       0.77      0.77      0.77     16512
weighted avg       0.77      0.77      0.77     16512



In [9]:
import pickle

with open('../models/win_predictor_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Also save the column names - we'll need this exact structure later
with open('../models/model_columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print(f"✅ Saved {best_model_name} model successfully")

✅ Saved XGBoost model successfully


In [10]:
import pickle

# Load the saved model back and test it
with open('../models/win_predictor_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Quick sanity test prediction
test_pred = loaded_model.predict_proba(X_test.iloc[:5])
print("Sample probability predictions:")
for i, pred in enumerate(test_pred):
    print(f"Row {i+1}: Loss probability: {pred[0]:.2f}, Win probability: {pred[1]:.2f}")

print("\n✅ Model loaded and working correctly")

Sample probability predictions:
Row 1: Loss probability: 0.00, Win probability: 1.00
Row 2: Loss probability: 0.00, Win probability: 1.00
Row 3: Loss probability: 0.00, Win probability: 1.00
Row 4: Loss probability: 0.00, Win probability: 1.00
Row 5: Loss probability: 0.00, Win probability: 1.00

✅ Model loaded and working correctly
